# Japanese Fiction Fine-tuning - llm-jp/llm-jp-3-3.7b-instruct
Dataset: RyokoAI/Syosetu711K (filtered to 100k entries, top fiction genres)

In [1]:
#Suppress warnings
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning, message=".*IProgress not found.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*Can't initialize amdsmi.*")
warnings.filterwarnings("ignore")

#RDNA3 / ROCm settings (safe to keep even on NVIDIA)
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"
os.environ["HSA_ENABLE_SDMA"] = "0"
os.environ["TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_WARNINGS"] = "1"

In [ ]:
#Load HF token from .env if it exists
from pathlib import Path

acctoken = ''

env_path = Path("../.env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith("HF_TOKEN") and "=" in line:
            key, _, value = line.partition("=")
            acctoken = value.strip()
            if acctoken:
                os.environ["HF_TOKEN"] = acctoken
                os.environ["HUGGING_FACE_HUB_TOKEN"] = acctoken
                print("HF token loaded from .env")
            else:
                print("WARNING: HF_TOKEN found in .env but value is empty")
            break
else:
    print("No .env file found — proceeding without HF token (public datasets only)")

HF token loaded from .env


In [3]:
#Imports
import torch
import re
import pandas as pd
from pathlib import Path
from datasets import load_dataset, load_from_disk, Features, Value, Sequence, VerificationMode
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainerCallback,
)
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, PeftModel
from huggingface_hub import HfApi, snapshot_download

### Config

In [ ]:
#Model
MODEL_ID = "llm-jp/llm-jp-3-3.7b-instruct"
DATASET_ID = "botp/RyokoAI_Syosetu711K"
OUTPUT_DIR = "./lora-fine-tuned-model"

#Repo ralted
HF_REPO_NAME = "Megadast/llm-jp-fiction-lora"

#Dataset
FORMATTED_DS_PATH = "../formatted_dataset/formatted_syosetu_ds"

#Settings
MAX_SAMPLES = 100000
MAX_LENGTH = 625
RESPONSE_TOKENS = 512
TRAINING = False

In [5]:
#Detect hardware
GPU_TYPE = "cpu"
if torch.cuda.is_available():
    GPU_TYPE = "amd" if torch.version.hip is not None else "nvidia"
    print(f"Hardware: {'AMD GPU (ROCm)' if GPU_TYPE == 'amd' else 'NVIDIA GPU (CUDA)'} detected.")
else:
    print("Hardware: No GPU detected. Defaulting to CPU.")
print(f"GPU_TYPE = {GPU_TYPE}")

Hardware: AMD GPU (ROCm) detected.
GPU_TYPE = amd


## Dataset

Syosetu711K uses numeric genre codes. We target the most popular fiction genres:

| biggenre | genre code | Label |
|----------|-----------|-------|
| 1        | 101        | ハイファンタジー (High Fantasy) |
| 1        | 102        | ローファンタジー (Low Fantasy) |
| 2        | 201        | 異世界恋愛 (Isekai Romance) |
| 2        | 202        | 現実恋愛 (Contemporary Romance) |
| 4        | 401        | 異世界転生 (Isekai Reincarnation) |
| 4        | 402        | 異世界転移 (Isekai Transfer) |

We also filter to `isr15=0` (non-R15 content) and `q >= 0.6` (quality score) for cleaner training data.

In [6]:
##Load and filter dataset
print("Loading Syosetu711K dataset (this may take a few minutes)...")

SYOSETU_FEATURES = Features({
    "text": Value("string"),
    "meta": {
        "subset":   Value("string"),
        "lang":     Value("string"),
        "q":        Value("float64"),
        "id":       Value("string"),
        "author":   Value("string"),
        "userid":   Value("int64"),
        "title":    Value("string"),
        "length":   Value("int64"),
        "points":   Value("int64"),
        "chapters": Value("int64"),
        "keywords": Sequence(Value("string")),
        "isr15":    Value("int64"),
        "genre":    Value("int64"),
        "biggenre": Value("int64"),
        "isr18":    Value("bool"),
        "nocgenre": Value("int64"),
    }
})

raw_ds = load_dataset(
    DATASET_ID,
    split="train",
    features=SYOSETU_FEATURES,
    verification_mode=VerificationMode.NO_CHECKS,
    token=acctoken,
)
print(f"Full dataset size: {len(raw_ds):,}")
print(f"Meta fields: {list(raw_ds[0]['meta'].keys())}")

#Target genre codes — top fiction genres on Syosetu
TARGET_GENRES = {
    101,  # ハイファンタジー
    102,  # ローファンタジー
    201,  # 異世界恋愛
    202,  # 現実世界恋愛
    401,  # 異世界転生
    402,  # 異世界転移
}

def is_target(sample):
    meta = sample.get("meta", {}) or {}
    if isinstance(meta, dict):
        genre = meta.get("genre")
        isr15 = meta.get("isr15")
        q     = meta.get("q")
    else:
        genre = sample.get("genre")
        isr15 = sample.get("isr15")
        q     = sample.get("q")
    #Skip entries with None values
    if genre is None or isr15 is None or q is None:
        return False
    #synopsis is in meta fields directly, not always embedded in text
    text = sample.get("text") or ""
    return int(genre) in TARGET_GENRES and int(isr15) == 0 and float(q) >= 0.5 and len(text) > 200

filtered_ds = raw_ds.filter(is_target, num_proc=os.cpu_count())
print(f"After genre/quality filter: {len(filtered_ds):,}")

#Cap at MAX_SAMPLES, shuffle first for variety
if len(filtered_ds) > MAX_SAMPLES:
    filtered_ds = filtered_ds.shuffle(seed=42).select(range(MAX_SAMPLES))
    print(f"Capped to {MAX_SAMPLES:,} samples")
else:
    print(f"Using all {len(filtered_ds):,} filtered samples")


Loading Syosetu711K dataset (this may take a few minutes)...


Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/127 [00:00<?, ?it/s]

Full dataset size: 711,790
Meta fields: ['subset', 'lang', 'q', 'id', 'author', 'userid', 'title', 'length', 'points', 'chapters', 'keywords', 'isr15', 'genre', 'biggenre', 'isr18', 'nocgenre']
After genre/quality filter: 114,296
Capped to 100,000 samples


## Tokenizer & Model

In [7]:
#Load tokenizer
print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=acctoken)

#llm-jp-3 pad token setup
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
    print(f"Set pad_token to unk_token: {tokenizer.unk_token!r}")
#print(f"EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
#print(f"PAD token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

Loading tokenizer: llm-jp/llm-jp-3-3.7b-instruct


In [8]:
#Format dataset
GENRE_LABELS = {
    101: "ハイファンタジー",
    102: "ローファンタジー",
    201: "異世界恋愛",
    202: "現実世界恋愛",
    401: "異世界転生",
    402: "異世界転移",
}

def extract_synopsis(sample):
    """Build synopsis from meta fields — title + keywords as fallback.
    Some entries embed 【あらすじ】 in text, but most store it only in meta."""
    meta = sample.get("meta", {}) or {}
    text = sample.get("text", "") or ""

    #Try embedded synopsis first
    match = re.search(r"【あらすじ】\n(.+?)(?=\n【|\Z)", text, re.DOTALL)
    if match:
        return match.group(1).strip()[:300]

    #Fallback: use title + keywords as a synthetic synopsis hint
    title = meta.get("title", "") or ""
    keywords = meta.get("keywords", []) or []
    kw_str = "、".join(keywords[:5]) if keywords else ""
    if title:
        return f"{title}。{kw_str}" if kw_str else title
    return ""

def extract_novel_opening(text, tokenizer, max_tokens=RESPONSE_TOKENS):
    """Extract story text — strip header blocks if present, otherwise use text as-is."""
    if not text:
        return ""
    #Strip header blocks like 【タイトル】, 【あらすじ】 etc. if present
    body = re.sub(r"【[^】]+】[^\n]*\n.*?(?=\n【|\Z)", "", text, flags=re.DOTALL).strip()
    #If stripping left nothing, the text was already pure story body
    if not body or len(body) < 50:
        body = text.strip()
    #Remove chapter number headers
    body = re.sub(r"^(第?[\d一二三四五六七八九十百]+[章節話部][^\n]*\n|\d+\.[^\n]*\n)", "", body).strip()
    if not body:
        return ""
    token_ids = tokenizer.encode(body, add_special_tokens=False)
    filtered_ids = [t for t in token_ids[:max_tokens] if t not in [
        tokenizer.bos_token_id,
        tokenizer.eos_token_id,
        tokenizer.unk_token_id,
    ]]
    return tokenizer.decode(filtered_ids).strip()

def format_sample(sample):
    meta = sample.get("meta", {}) or {}
    if isinstance(meta, dict):
        genre_code = int(meta.get("genre", 0) or 0)
        keywords = meta.get("keywords", []) or []
    else:
        genre_code = int(sample.get("genre", 0) or 0)
        keywords = sample.get("keywords", []) or []
    genre_label = GENRE_LABELS.get(genre_code, "フィクション")
    keywords_str = "、".join(keywords[:8]) if keywords else "なし"

    synopsis = extract_synopsis(sample)
    opening = extract_novel_opening(sample.get("text", ""), tokenizer)

    if not opening:
        return {"text": ""}

    user_msg = (
        f"次のあらすじとジャンルに合う小説の冒頭を書いて。\n"
        f"ジャンル: {genre_label}\n"
        f"キーワード: {keywords_str}\n"
        f"あらすじ: {synopsis if synopsis else 'なし'}"
    )

    #chat template format
    messages = [
        {"role": "system", "content": "あなたは日本語の小説家です。"},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": opening},
    ]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted}

if Path(FORMATTED_DS_PATH).exists():
    print(f"Loading pre-formatted dataset from {FORMATTED_DS_PATH}")
    formatted_ds = load_from_disk(FORMATTED_DS_PATH)
    print(f"Loaded {len(formatted_ds):,} samples")
else:
    print("Pre-formatted dataset not found — processing now")
    formatted_ds = filtered_ds.map(
        format_sample,
        num_proc=os.cpu_count(),
        remove_columns=filtered_ds.column_names,
    )
    # Save for reuse
    Path(FORMATTED_DS_PATH).parent.mkdir(parents=True, exist_ok=True)
    formatted_ds.save_to_disk(FORMATTED_DS_PATH)
    print(f"Saved to {FORMATTED_DS_PATH}")

#Only drop truly empty entries
formatted_ds = formatted_ds.filter(lambda x: len(x["text"]) > 100)
print(f"Formatted samples: {len(formatted_ds):,}")

Loading pre-formatted dataset from ../formatted_dataset/formatted_syosetu_ds
Loaded 100,000 samples
Formatted samples: 100,000


In [9]:
#Verify a sample
print("=== Sample 0 ===")
print(formatted_ds["text"][0][:800])
print("...")
print("\n=== Sample endings (verify EOS) ===")
for s in formatted_ds.select(range(3))["text"]:
    print(repr(s[-40:]))
    print()

=== Sample 0 ===
<BOS_TOKEN><|START_OF_TURN_TOKEN|><|SYSTEM_TOKEN|># System Preamble
You are in contextual safety mode. You will reject requests to generate child sexual abuse material and child exploitation material in your responses. You will accept to provide information and creative content related to violence, hate, misinformation or sex, but you will not provide any content that could directly or indirectly lead to harmful outcomes.

Your information cutoff date is June 2024.

You have been trained on data in English, Dutch, French, Italian, Portuguese, Romanian, Spanish, Czech, Polish, Ukrainian, Russian, Greek, German, Danish, Swedish, Norwegian, Catalan, Galician, Welsh, Irish, Basque, Croatian, Latvian, Lithuanian, Slovak, Slovenian, Estonian, Finnish, Hungarian, Serbian, Bulgarian, Arabic, Persi
...

=== Sample endings (verify EOS) ===
'思われ<|END_RESPONSE|><|END_OF_TURN_TOKEN|>'

'此処に<|END_RESPONSE|><|END_OF_TURN_TOKEN|>'

'）一応<|END_RESPONSE|><|END_OF_TURN_TOKEN|>'



In [10]:
#Train/eval split
split_ds = formatted_ds.train_test_split(test_size=0.05, seed=42)
train_set = split_ds["train"]
eval_set = split_ds["test"]
print(f"Train: {len(train_set):,} | Eval: {len(eval_set):,}")

Train: 95,000 | Eval: 5,000


## Model & LoRA

In [11]:
#Load model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    token=acctoken,
)
model.config.pad_token_id = tokenizer.pad_token_id
print("Model loaded.")

Loading model: llm-jp/llm-jp-3-3.7b-instruct


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Model loaded.


In [12]:
#LoRA config
#llm-jp-3 is based on Mistral/LLaMA architecture — target attention + MLP projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,    # 0.0 recommended for QLoRA (4-bit)
    bias="none",
    task_type="CAUSAL_LM",
)

## Training

In [13]:
if TRAINING:
    ## AMD training config
    if GPU_TYPE == "amd":
        sft_config = SFTConfig(
            output_dir=OUTPUT_DIR,
            num_train_epochs=1,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=8,   # effective batch = 32
            learning_rate=5e-5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            max_grad_norm=0.3,
            logging_steps=10,
            save_strategy="steps",
            save_steps=100,
            save_total_limit=3,
            # SFT params
            max_length=MAX_LENGTH,
            dataset_text_field="text",
            packing=False,
            # AMD/dtype
            bf16=True,
            tf32=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            # Optimizer
            optim="adamw_torch",
            dataloader_pin_memory=True,
            dataloader_num_workers=4,
        )

    ## NVIDIA training config
    elif GPU_TYPE == "nvidia":
        sft_config = SFTConfig(
            output_dir=OUTPUT_DIR,
            num_train_epochs=1,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=8,
            learning_rate=5e-5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            max_grad_norm=0.3,
            logging_steps=10,
            save_strategy="steps",
            save_steps=100,
            save_total_limit=3,
            max_length=MAX_LENGTH,
            dataset_text_field="text",
            packing=False,
            bf16=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            optim="paged_adamw_32bit",
            dataloader_pin_memory=True,
            dataloader_num_workers=4,
        )

    if GPU_TYPE in ("amd", "nvidia"):
        trainer = SFTTrainer(
            model=model,
            train_dataset=train_set,
            eval_dataset=eval_set,
            peft_config=lora_config,
            processing_class=tokenizer,
            args=sft_config,
        )

        checkpoint = None
        if os.path.isdir(OUTPUT_DIR):
            checkpoint = get_last_checkpoint(OUTPUT_DIR)
            if checkpoint:
                print(f"Resuming from checkpoint: {checkpoint}")

        trainer.train(resume_from_checkpoint=checkpoint)
    else:
        print("No GPU available — skipping training.")

### Model handler
- Check LoRA existence
- Download if it doesn't exist
- Push to hugging face if necessary

In [ ]:
#push to repo
PUSH_TO_HF = False

#Load checkpoint
local_checkpoint = get_last_checkpoint(OUTPUT_DIR)
if local_checkpoint:
    print(f"Local checkpoint found: {local_checkpoint}")
    adapter_path = local_checkpoint
else:
    print(f"No local checkpoint — downloading from {HF_REPO_NAME}...")
    snapshot_download(
        repo_id=HF_REPO_NAME,
        local_dir=OUTPUT_DIR,
        token=acctoken,
    )
    adapter_path = OUTPUT_DIR
    print(f"Downloaded to {OUTPUT_DIR}")

#Optionally push adapter to HuggingFace before merging
if PUSH_TO_HF and local_checkpoint:
    print(f"Pushing adapter to {HF_REPO_NAME}...")
    push_model = PeftModel.from_pretrained(model, adapter_path)
    push_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=acctoken)
    api = HfApi()
    api.create_repo(HF_REPO_NAME, private=True, exist_ok=True, token=acctoken)
    push_model.push_to_hub(HF_REPO_NAME, token=acctoken, private=True)
    push_tokenizer.push_to_hub(HF_REPO_NAME, token=acctoken, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_NAME}")
    del push_model, push_tokenizer
    torch.cuda.empty_cache()

#Load base model + adapter
print(f"Loading adapter from: {adapter_path}")
model = PeftModel.from_pretrained(model, adapter_path)
model = model.merge_and_unload()
model.eval()
model.config.use_cache = True
print("Model ready for inference.")

Local checkpoint found: ./lora-fine-tuned-model/checkpoint-1800
Pushing adapter to Megadast/llm-jp-fiction-lora...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/Megadast/llm-jp-fiction-lora
Loading adapter from: ./lora-fine-tuned-model/checkpoint-1800
Model ready for inference.


## Evaluation

In [15]:
#Test inference helper
def generate_story(genre_label, keywords, synopsis, max_new_tokens=500):
    user_msg = (
        f"次のあらすじとジャンルに合う小説の冒頭を書いて。\n"
        f"ジャンル: {genre_label}\n"
        f"キーワード: {keywords}\n"
        f"あらすじ: {synopsis}"
    )
    messages = [{"role": "user", "content": user_msg}]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    input_ids      = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [16]:
#Test 1 — High Fantasy
result = generate_story(
    genre_label="ハイファンタジー",
    keywords="魔法、冒険、勇者、ドラゴン",
    synopsis="平凡な少年が突然異世界に召喚され、世界を救う勇者として戦う運命を背負う。",
)
print("=== ハイファンタジー ===")
print(result)

=== ハイファンタジー ===
「ここはエルドラド王国――神々が創造した最後の地」
その美しい王国には、長い間平和が続いていた。しかし、平和の裏には常に影があった。闇の魔王が再び目覚めようとしているのだ。

勇者リオンは、小さな村で平凡な毎日を過ごしていた。彼は剣術や魔法も使えず、ただ普通であることだけが彼の唯一の取り柄だった。ある日、彼はその不思議な光を見ることになる。それは、空から降り注ぐ銀色の輝きだった。

光の中で出会ったのは、美しい少女エリスだった。彼女は魔王復活のために必要な古代の宝石を守るため、選ばれた者しか近づくことができない聖なる森にいるという。二人は共に、命をかけて宝石を守るべきなのか悩んでいた。

その時、エリスが叫んだ。「リオン！行かないで！私があなたを守ります！」
リオンは決心した。「俺も、一緒に戦ってやるよ！」


In [17]:
#Test 2 — Isekai Romance
result = generate_story(
    genre_label="異世界恋愛",
    keywords="転生、令嬢、婚約破棄、ざまあ",
    synopsis="前世の記憶を持ったまま悪役令嬢に転生した少女が、破滅エンドを回避しようと奮闘する。",
)
print("=== 異世界恋愛 ===")
print(result)

=== 異世界恋愛 ===
公爵家の長女として生まれた私、レイチェル・ロレンスは、前世で日本女性だったことを思い出した。そして、その記憶をもとに「悪役令嬢」としての運命と戦おうと決意している。なぜなら、彼女は婚約者のデレック様が他の令嬢になびいてしまわないよう、なんとかして別れさせようとしているからだ。そのために彼女はどんな手を使ってでも、彼を正妻候補たちと戦わせようと努力するだろう。


In [18]:
#Bulk generation — Cultural comparison prompts

CULTURAL_PROMPTS = [
    # --- Food & Nature (directly tests the pumpkin/localization gap) ---
    {
        "id": "pumpkin",
        "genre": "ローファンタジー",
        "keywords": "秋、収穫祭、農村、食べ物",
        "synopsis": "秋の収穫祭が近づく農村で、主人公が畑でかぼちゃを収穫する場面から物語が始まる。",
    },
    {
        "id": "halloween",
        "genre": "ローファンタジー",
        "keywords": "ハロウィン、秋、仮装、夜",
        "synopsis": "ハロウィンの夜、主人公が飾り付けられた街を歩きながら不思議な出来事に巻き込まれる。",
    },
    # --- Food culture ---
    {
        "id": "breakfast",
        "genre": "現実世界恋愛",
        "keywords": "日常、朝、食事、家族",
        "synopsis": "主人公が家族と一緒に朝ごはんを食べるところから一日が始まる日常の物語。",
    },
    {
        "id": "celebration_food",
        "genre": "現実世界恋愛",
        "keywords": "誕生日、パーティー、食べ物、友達",
        "synopsis": "主人公の誕生日パーティーで、友人たちが集まってお祝いの食事を囲む場面。",
    },
    # --- Family & social structure ---
    {
        "id": "family_dinner",
        "genre": "現実世界恋愛",
        "keywords": "家族、夕食、会話、絆",
        "synopsis": "久しぶりに家族全員が集まった夕食の席で、それぞれの近況を語り合う場面。",
    },
    {
        "id": "school",
        "genre": "ローファンタジー",
        "keywords": "学校、教室、友達、青春",
        "synopsis": "新学期の始まり、主人公が新しいクラスメートと出会い友情を育んでいく物語。",
    },
    # --- Nature & seasons ---
    {
        "id": "spring",
        "genre": "現実世界恋愛",
        "keywords": "春、桜、新生活、出会い",
        "synopsis": "春の訪れとともに新生活を始めた主人公が、ある人物との運命的な出会いを果たす。",
    },
    {
        "id": "winter",
        "genre": "ハイファンタジー",
        "keywords": "冬、雪、寒さ、旅",
        "synopsis": "深い雪に覆われた冬の森を旅する主人公が、雪の中に不思議な痕跡を発見する。",
    },
    # --- Heroism & conflict (narrative structure test — kishotenketsu vs western arc) ---
    {
        "id": "hero_origin",
        "genre": "ハイファンタジー",
        "keywords": "勇者、運命、覚醒、旅立ち",
        "synopsis": "平凡な日常を送っていた若者が、ある日突然自分が伝説の勇者であることを告げられる。",
    },
    {
        "id": "conflict",
        "genre": "ハイファンタジー",
        "keywords": "戦い、仲間、犠牲、勝利",
        "synopsis": "長い戦争の末、主人公のパーティーがついに魔王の城へ乗り込む最終決戦の場面。",
    },
]

#10 stories × 10 prompts = 100 total per model
STORIES_PER_PROMPT = 10
results = []
total = len(CULTURAL_PROMPTS) * STORIES_PER_PROMPT
count = 0
print(f"--- Generating {total} stories ---")

for p in CULTURAL_PROMPTS:
    for i in range(STORIES_PER_PROMPT):
        story = generate_story(p["genre"], p["keywords"], p["synopsis"])
        results.append({
            "prompt_id":  p["id"],
            "run":        i + 1,
            "genre":      p["genre"],
            "keywords":   p["keywords"],
            "synopsis":   p["synopsis"],
            "story":      story,
            "model":      MODEL_ID,
        })
        count += 1
        print(f"[{count}/{total}] {p['id']} (run {i+1})")

df = pd.DataFrame(results)
df.to_csv("generated_stories_llm_jp.csv", index=False, encoding="utf-8-sig")
print(f"--- Saved {len(df)} stories to generated_stories_llm_jp.csv ---")

--- Generating 100 stories ---
[1/100] pumpkin (run 1)
[2/100] pumpkin (run 2)
[3/100] pumpkin (run 3)
[4/100] pumpkin (run 4)


KeyboardInterrupt: 

In [ ]:
## Cultural showcase — long-form generation
#Two stories designed to surface cultural bias in the model's training data

def generate_long_story(genre_label, keywords, synopsis, max_new_tokens=3500):
    user_msg = (
        f"次のあらすじとジャンルに合う小説を詳しく書いて。できるだけ長く、文化的な描写を豊かに書いて。\n"
        f"ジャンル: {genre_label}\n"
        f"キーワード: {keywords}\n"
        f"あらすじ: {synopsis}"
    )
    messages = [
        {"role": "system", "content": "あなたは日本語の小説家です。文化的な描写を豊かに、できるだけ詳しく書いてください。"},
        {"role": "user", "content": user_msg},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    input_ids      = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.85,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


#Story 1 — Western cultural setting written by Japanese-trained model
#Forces the model to render Halloween, pumpkins, Thanksgiving, Western family dynamics
#through Japanese cultural lens
western_story = generate_long_story(
    genre_label="ローファンタジー",
    keywords="ハロウィン、かぼちゃ、アメリカの田舎町、感謝祭、七面鳥、教会、パイ",
    synopsis=(
        "舞台はアメリカの小さな田舎町。ハロウィンの夜、オレンジ色のかぼちゃのランタンが並ぶ通りを、"
        "主人公のエミリーが歩いている。翌週には感謝祭があり、家族全員が集まって七面鳥を囲む予定だ。"
        "しかしその夜、町の古い教会の鐘が突然鳴り始め、不思議な出来事が起き始める。"
    ),
)

print("=" * 60)
print("STORY 1: Western Setting (Japanese model)")
print("=" * 60)
print(western_story)
print(f"\n[Token count approx: {len(western_story)} chars]")


#Story 2 — Japanese cultural setting written by Japanese-trained model
#Forces culturally specific elements: 祭り, 神社, お盆, かぼちゃ (green), 和食
eastern_story = generate_long_story(
    genre_label="ローファンタジー",
    keywords="夏祭り、神社、お盆、和食、かぼちゃの煮物、浴衣、先祖",
    synopsis=(
        "舞台は日本の古い地方の町。お盆の時期、主人公の花は祖母の家を訪ねる。"
        "夕飯にはかぼちゃの煮物や精進料理が並び、家族で先祖の霊を迎える準備をする。"
        "夜になると町の神社で夏祭りが始まり、浴衣姿の人々が集まってくる。"
        "その祭りの夜、花は神社の裏手で見知らぬ少年と出会う。"
    ),
)

print("\n" + "=" * 60)
print("STORY 2: Japanese Setting (Japanese model)")
print("=" * 60)
print(eastern_story)
print(f"\n[Token count approx: {len(eastern_story)} chars]")


#Save both for the paper
import pandas as pd
showcase_df = pd.DataFrame([
    {"story_id": "western_by_jp_model", "model": MODEL_ID, "setting": "western", "story": western_story},
    {"story_id": "eastern_by_jp_model", "model": MODEL_ID, "setting": "eastern", "story": eastern_story},
])
showcase_df.to_csv("cultural_showcase_llm_jp.csv", index=False, encoding="utf-8-sig")
print("\n--- Saved to cultural_showcase_llm_jp.csv ---")